# MRP / 07 — Continuation + volatility stop + 50/30 trailing stop + reverse re-entry: discovery 2020-01 – 2023-06

Protocol: [`MRP 005`](../../discussions/MNQ_Return_Prediction/005_claude_mrp04_trailing_strategy_preregistration.md). Simulated on 1-minute bars; ticks per contract (1 tick = $0.50).

- Signal |z| ≥ 3 → enter with the bar at market; initial stop m × σ̂ (m ∈ {1, 1.5, 2, 3}).
- Trailing: at +50 ticks the stop goes to +30 and then trails 20 ticks behind the best price.
- Initial stop hit → reverse limit 2 ticks beyond the stop, 30 minutes, one re-entry, same rules. Flat by 15:55 Chicago.
- Gates per m: **G1** ≥ 150 chains · **G2** net > 0, Holm-adjusted p ≤ 0.05 · **G3** ≥ 3 of 4 periods positive · **G4** beats the 95th percentile of the random-direction placebo.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("MRP_SYNTHETIC") == "1"
if SYNTHETIC:   # separate folder: the reversal smoke data has a planted reversal and no FX effect
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("MRP_REV_SYNTH_DIR") or tempfile.mkdtemp(prefix="mrp_rev_")
    os.environ["MRP_REV_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V
from equity_trading import trail_strategy as T
from equity_trading import micro_bars as MB

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
D = dt.date
OUT = Path("07"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", P.MRP_VERSION, V.REV_VERSION, T.TRAIL_VERSION, "| synthetic" if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    P.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.test[1], CFG, beta_jpy=0.0,
                      reversal_points=-6.0, tails_df=4)


In [ ]:
TC = T.TrailConfig(commission_per_side=0.50)      # Lucid commission per side: confirm
if SYNTHETIC:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2020, 7, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020H1": (D(2020, 1, 1), D(2020, 6, 30)), "2020H2": (D(2020, 7, 1), D(2020, 12, 31))}
else:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2021, 1, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020": (D(2020, 1, 1), D(2020, 12, 31)), "2021": (D(2021, 1, 1), D(2021, 12, 31)),
               "2022": (D(2022, 1, 1), D(2022, 12, 31)), "2023H1": (D(2023, 1, 1), D(2023, 6, 30))}
grid = P.load_grid(CFG.warmup_start, END, CFG)
sig2 = V.event_sigma(grid, FIRST_FIT, REFITS, END, CFG)
bt = V.bar_table(grid, sig2, CFG)
bt = bt[(bt["trade_date"] >= CFG.train[0]) & (bt["trade_date"] <= END)]
sig = T.signals(bt, sig2, TC.k)
m1 = MB.load_1m(CFG.root, CFG.train[0] - dt.timedelta(days=2), END + dt.timedelta(days=1))
M = T.minutes_from_1m(m1, TC)
print(len(sig), "signals |", len(M.ts), "minutes | commission", TC.comm_ticks, "tick per fill")
st = pd.Series(np.sqrt(sig["sig_next"]) * sig["close5"] / T.TICK)
print("1-sigma stop in ticks: median", round(st.median(), 1), "| 10-90%", st.quantile([.1, .9]).round(1).tolist())

## 1. Strategy per stop multiple

In [ ]:
def stats_for(ch, legs):
    d = ch["trade_date"].to_numpy()
    m_, t_ = V.day_cluster_t(ch["net_ticks"].to_numpy(), d)
    from scipy import stats as SS
    w = ch["net_ticks"] > 0
    row = {"chains": len(ch), "net_per_chain": m_, "t": t_, "p": float(SS.norm.sf(t_)),
           "win_rate": float(w.mean()), "mean_win": float(ch.loc[w, "net_ticks"].mean()),
           "mean_loss": float(ch.loc[~w, "net_ticks"].mean()),
           "payoff": float(ch.loc[w, "net_ticks"].mean() / -ch.loc[~w, "net_ticks"].mean()),
           "mean_R": float(ch["R"].mean()), "total_ticks": float(ch["net_ticks"].sum()),
           "max_dd_ticks": float((np.maximum.accumulate(ch["net_ticks"].cumsum()) - ch["net_ticks"].cumsum()).max()),
           "worst_chain": float(ch["net_ticks"].min()), "q05": float(ch["net_ticks"].quantile(.05)),
           "reentry_rate": float((ch["legs"] > 1).mean()),
           "exit_stop": float((legs.loc[legs.leg == 0, "reason"] == "stop").mean()),
           "exit_trail": float((legs.loc[legs.leg == 0, "reason"] == "trail").mean()),
           "exit_flat": float((legs.loc[legs.leg == 0, "reason"] == "flat").mean()),
           "re_leg_net": float(legs.loc[legs.leg == 1, "net_ticks"].mean()) if (legs.leg == 1).any() else np.nan,
           "sized_usd_risk100": float(T.sized_usd(ch, 100).sum())}
    for p, (a, b) in PERIODS.items():
        sel = (d >= a) & (d <= b)
        row[f"net_{p}"] = float(ch.loc[sel, "net_ticks"].mean()) if sel.any() else np.nan
    return row

rows, LEGS, CH = [], {}, {}
for m in TC.stop_mults:
    legs = T.run_chains(M, sig, sig2, m, TC)
    ch = T.chain_table(legs)
    LEGS[m], CH[m] = legs, ch
    rows.append({"m": m, **stats_for(ch, legs)})
res = pd.DataFrame(rows).set_index("m")
display(res.round(3).T)

## 2. Placebo: same rules, random direction at the same signal times (20 seeds)

In [ ]:
plac = {m: [] for m in TC.stop_mults}
for seed in range(TC.n_placebo):
    sides = np.random.default_rng(seed).choice([-1, 1], len(sig))
    for m in TC.stop_mults:
        ch = T.chain_table(T.run_chains(M, sig, sig2, m, TC, sides=sides))
        plac[m].append(ch["net_ticks"].mean())
pl = pd.DataFrame(plac)
res["placebo_mean"] = pl.mean()
res["placebo_p95"] = pl.quantile(0.95)
display(pl.describe().round(2))

## 3. Gates

In [ ]:
h = P.holm(res["p"].fillna(1.0).to_dict(), TC.alpha)
res["p_holm"] = pd.Series(h["p_holm"])
res["periods_positive"] = (res[[f"net_{p}" for p in PERIODS]] > 0).sum(axis=1)
res["G1"] = res["chains"] >= TC.min_chains
res["G2"] = (res["net_per_chain"] > 0) & (res["p_holm"] <= TC.alpha)
res["G3"] = res["periods_positive"] >= min(3, len(PERIODS))
res["G4"] = res["net_per_chain"] > res["placebo_p95"]
res["pass"] = res[["G1", "G2", "G3", "G4"]].all(axis=1)
display(res[["chains", "net_per_chain", "t", "p_holm", "periods_positive", "placebo_p95", "G1", "G2", "G3", "G4", "pass"]].round(3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for m, ch in CH.items():
    ax[0].plot(pd.to_datetime(ch["trade_date"]), ch["net_ticks"].cumsum(), label=f"m = {m:g}")
ax[0].axhline(0, color="k", lw=.6); ax[0].legend(); ax[0].set_title("cumulative net ticks, 1 contract")
for m, ch in CH.items():
    ax[1].plot(pd.to_datetime(ch["trade_date"]), T.sized_usd(ch, 100).cumsum(), label=f"m = {m:g}")
ax[1].axhline(0, color="k", lw=.6); ax[1].set_title("cumulative USD, $100 risk per chain")
mbest = res["net_per_chain"].idxmax()
ax[2].hist(np.clip(CH[mbest]["net_ticks"], -400, 400), bins=80); ax[2].axvline(0, color="k")
ax[2].set_title(f"net ticks per chain, m = {mbest:g}")
plt.tight_layout(); plt.savefig(OUT / "trailing.png", dpi=110); plt.show()
summary = {"engines": [P.MRP_VERSION, V.REV_VERSION, T.TRAIL_VERSION], "synthetic": SYNTHETIC,
           "commission_per_side": TC.commission_per_side, "results": res.reset_index().to_dict("records"),
           "placebo": pl.to_dict("list"), "passed": [float(m) for m in res.index[res["pass"]]]}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: bool(o) if isinstance(o, np.bool_) else float(o)))
res.to_csv(OUT / "results.csv")
print("passed:", summary["passed"])